# Decoding

Select a saved pairing below. Results are read from `out/vs4_decoding/<pairing>/`; compatible archived runs are indexed for comparisons across methods, classifiers, and components. The JSON supplies the default display settings but is never overwritten.

Figure layouts, colors, statistics, and plotting code remain in the notebook. No raw recordings or trial caches are loaded. Legacy cleaned pickle files are ignored; the current CSV and paired NPZ are the authoritative inputs. Missing combinations remain unavailable, not zero accuracy.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import re
import hashlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import OrderedDict
from matplotlib.lines import Line2D
from sklearn.metrics import accuracy_score, f1_score
import sys
from pathlib import Path
from IPython.display import display

ROOT = Path(os.environ.get('MIB_PROJECT_ROOT', Path.cwd())).expanduser().resolve()
if not (ROOT / 'vs4_1_run_decoding.py').is_file():
    ROOT = ROOT / 'iEEGvsMEG'
if not (ROOT / 'vs4_1_run_decoding.py').is_file():
    raise FileNotFoundError('Start in the repository or iEEGvsMEG folder, or set MIB_PROJECT_ROOT.')
sys.path.insert(0, str(ROOT))
from vs0_run_analysis import read_config
from vs0_utils_pairing import PAIRING_IDS

OUT_ROOT = Path(os.environ.get('MIB_OUTPUT_ROOT', ROOT / 'out')).expanduser().resolve()
CONFIG_FILE = OUT_ROOT / 'vs4_decoding' / 'config.json'
CONFIG = read_config(CONFIG_FILE)
PARAMETERS = CONFIG['parameters']

# None follows the config when it selects one pairing; otherwise displays match.
# Or set an explicit value: match, order, random_001 ... random_020.
PAIRING_ID = None
INCLUDE_ARCHIVED_RUNS = True
STRICT_MISSING_RESULTS = False
USE_CLEANED_CACHE = True
RECOMPUTE_CLEANED_DECODING = False
MAX_IN_MEMORY_RESULTS = 2

selection = CONFIG.get('pairings', 'match')
if PAIRING_ID is None:
    PAIRING_ID = selection if isinstance(selection, str) and selection != 'all' else (selection[0] if isinstance(selection, list) and len(selection) == 1 else 'match')
if PAIRING_ID not in PAIRING_IDS:
    raise ValueError('Select a known saved pairing.')
DECODING_ROOT = OUT_ROOT / 'vs4_decoding'
DECODING_RESULTS_DIR = DECODING_ROOT / PAIRING_ID
modality = {'ieeg': 'navy', 'meg': 'orange'}
print('Displayed pairing:', PAIRING_ID)
print('Current result folder:', DECODING_RESULTS_DIR)
display(pd.json_normalize(PARAMETERS, sep='.').T.rename(columns={0: 'configured value'}))

In [ ]:
# Current writer layout: predictions are serialized as iteration × permutation × condition.
# Cleaned caches retain the small arrays required by the figures, not all shuffled weights.
CLEANED_FORMAT_VERSION = 1
_RESULT_CACHE = OrderedDict()
_MISSING_RESULTS = set()


def source_signature(path):
    path = Path(path)
    stat = path.stat()
    return {'path': str(path.resolve()), 'size': stat.st_size, 'mtime_ns': stat.st_mtime_ns}


def numbers(value):
    if not isinstance(value, str):
        return np.asarray(value, dtype=float).ravel()
    if '...' in value or '…' in value:
        raise ValueError('Serialized array is truncated; obtain the complete CSV before plotting.')
    value = re.sub(r'(?:np|numpy)\.(?:u?int(?:8|16|32|64)?|float(?:16|32|64|128)?)\s*\(', '(', value)
    pattern = r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?|[-+]?(?:nan|inf)'
    return np.asarray([float(token) for token in re.findall(pattern, value, flags=re.I)])


def read_network(path):
    with np.load(path, allow_pickle=False) as saved:
        required = {'times', 'decoding_time_ids', 'train_condition_mean', 'pc_use', 'method', 'conditions'}
        if not required <= set(saved.files):
            raise ValueError(f'{path}: missing current time-course metadata.')
        network = {key: saved[key].copy() for key in required}
    network['times'] = np.asarray(network['times'], dtype=float).ravel()
    raw_ids = np.asarray(network['decoding_time_ids']).ravel()
    if not np.isfinite(raw_ids).all() or not np.equal(raw_ids, np.floor(raw_ids)).all():
        raise ValueError(f'{path}: invalid decoding time indices.')
    network['decoding_time_ids'] = raw_ids.astype(int)
    ids, times = network['decoding_time_ids'], network['times']
    if not len(ids) or (ids < 0).any() or (ids >= len(times)).any() or not np.all(np.diff(ids) > 0):
        raise ValueError(f'{path}: invalid decoding window.')
    if not np.isfinite(times).all() or not np.all(np.diff(times) > 0):
        raise ValueError(f'{path}: invalid full-epoch time axis.')
    if network['train_condition_mean'].shape != (2, len(times)) or len(network['conditions']) != 2:
        raise ValueError(f'{path}: expected two full-epoch network means.')
    network['pc_use'] = int(network['pc_use'])
    network['method'] = str(network['method'].item()).upper()
    return network


def discover_results():
    candidates, excluded = [], []
    run_folders = [DECODING_RESULTS_DIR]
    if INCLUDE_ARCHIVED_RUNS:
        run_folders += sorted((DECODING_ROOT / 'archive').glob(PAIRING_ID + '_*'))
    compare_keys = ['band', 'method_pls', 'data_aug_method', 'bs_decoding', 'meg_kind', 'block_scaling', 'n_tr', 'seed']
    for run in run_folders:
        manifest_path = run / 'manifest.json'
        if not manifest_path.is_file():
            continue
        manifest = json.loads(manifest_path.read_text())
        if manifest.get('status') != 'complete' or manifest.get('pairing') != PAIRING_ID:
            excluded.append((str(run), 'Incomplete run or different pairing'))
            continue
        settings = manifest.get('configuration', {}).get('parameters', {})
        differences = [key for key in compare_keys if settings.get(key) != PARAMETERS.get(key)]
        if differences:
            excluded.append((str(run), 'Different preprocessing/design: ' + ', '.join(differences)))
            continue
        identity = {'pairs': manifest.get('provenance', {}).get('pairs', {}),
                    'trial_cache': manifest.get('provenance', {}).get('trial_cache', {}).get('manifest_sha256')}
        for dataset in modality:
            for csv in sorted((run / dataset / str(PARAMETERS['band']) / 'fullmodel').glob('*.csv')):
                npz = csv.with_name(csv.stem + '_iteration0_time_courses.npz')
                if not npz.is_file():
                    continue
                labels = pd.read_csv(csv, usecols=lambda c: c in {'band', 'method', 'method_pca', 'data_aug_method', 'iter', 'iter_perm'})
                if len(labels) != 1 or not {'band', 'method', 'method_pca', 'data_aug_method'} <= set(labels):
                    raise ValueError(f'{csv}: expected one current decoding summary row.')
                row = labels.iloc[0]
                network = read_network(npz)
                method, component = str(row['method']).upper(), network['pc_use']
                if method != network['method']:
                    raise ValueError(f'{csv}: method differs from time-course artifact.')
                prefix = f"{method}_{row['method_pca']}_{row['data_aug_method']}_{component}_"
                if not csv.stem.startswith(prefix):
                    raise ValueError(f'{csv}: filename and recorded component disagree.')
                tail = csv.stem[len(prefix):]
                match = re.fullmatch(r'(LR|RandomForest|SVC_linear|SVC_rbf)full(?:_crp(?:None|-?\d+)_(?:None|-?\d+)_)?', tail)
                if match is None:
                    raise ValueError(f'{csv}: cannot identify classifier from filename.')
                expected_ids = np.arange(len(network['times']))
                if PARAMETERS.get('bs_decoding', False):
                    expected_ids = expected_ids[:np.searchsorted(network['times'], -.5, side='right')]
                if not np.array_equal(network['decoding_time_ids'], expected_ids):
                    excluded.append((str(csv), 'Different decoding window'))
                    continue
                key = (dataset, method, str(row['method_pca']), str(row['data_aug_method']), component, match[1])
                candidates.append(dict(key=key, csv=csv, npz=npz, network=network, labels=labels,
                    run=run, current=run == DECODING_RESULTS_DIR, identity=identity,
                    priority=(run == DECODING_RESULTS_DIR, csv.stat().st_mtime_ns, manifest_path.stat().st_mtime_ns)))
    candidates.sort(key=lambda item: item['priority'], reverse=True)
    chosen = {}
    # Current outputs take precedence; archived methods/components must share inputs.
    anchor_identity = candidates[0]['identity'] if candidates else None
    for record in candidates:
        if record['identity'] != anchor_identity:
            excluded.append((str(record['csv']), 'Different pairing files or trial-cache provenance'))
            continue
        if record['key'] not in chosen:
            chosen[record['key']] = record
    return chosen, excluded


RESULTS, EXCLUDED_RESULTS = discover_results()
AVAILABLE_RESULTS = pd.DataFrame([dict(dataset=k[0], method=k[1], representation=k[2], augmentation=k[3],
    pc_use=k[4], classifier=k[5], source='current' if v['current'] else 'archive', file=str(v['csv'])) for k, v in RESULTS.items()])
print(f'{len(RESULTS)} compatible decoding summaries available for {PAIRING_ID}.')
display(AVAILABLE_RESULTS)
if EXCLUDED_RESULTS:
    display(pd.DataFrame(EXCLUDED_RESULTS, columns=['excluded run/file', 'reason']))
if not RESULTS:
    print('No completed results found. Run the JSON command with the selected pairing before plotting.')


def clean_current_csv(record):
    columns = {'iter', 'iter_perm', 'y_pred', 'y_pred_sh', 'y_test', 'weight', 'weight_sh'}
    frame = pd.read_csv(record['csv'], usecols=lambda c: c in columns)
    if len(frame) != 1 or not columns <= set(frame):
        raise ValueError(f"{record['csv']}: incomplete decoding summary.")
    row = frame.iloc[0]
    iterations, permutations = int(row['iter']), int(row['iter_perm'])
    if iterations < 1 or permutations < 0:
        raise ValueError('Invalid decoding iteration/permutation counts.')
    truth, prediction, shuffled = (numbers(row[k]) for k in ('y_test', 'y_pred', 'y_pred_sh'))
    if len(truth) != 2*iterations or len(prediction) != len(truth) or len(shuffled) != 2*iterations*permutations:
        raise ValueError('Prediction counts do not match the recorded iterations/permutations.')
    if not np.isin(truth, [0, 1]).all() or not np.isin(prediction, [0, 1]).all() or not np.isin(shuffled, [0, 1]).all():
        raise ValueError('Saved classifier labels must be 0 or 1.')
    # Writer order is iteration -> shuffled model -> two held-out conditions.
    shuffled = shuffled.reshape(iterations, permutations, 2).transpose(0, 2, 1).reshape(2*iterations, permutations)
    nt = len(record['network']['decoding_time_ids'])
    classifier = record['key'][-1]
    has_weights = classifier != 'SVC_rbf'
    weights = np.empty((0, nt))
    control_mean = control_std = control_95 = np.full(nt, np.nan)
    if has_weights:
        values = numbers(row['weight'])
        if len(values) != iterations*nt:
            raise ValueError('Classifier-weight length differs from the saved decoding window.')
        weights = values.reshape(iterations, nt)
        if permutations:
            values = numbers(row['weight_sh'])
            if len(values) != iterations*permutations*nt:
                raise ValueError('Shuffled classifier-weight count differs from the saved window.')
            control = values.reshape(iterations*permutations, nt)
            control_mean, control_std = control.mean(0), control.std(0)
            control_95 = np.percentile(control, 95, axis=0)
            del control, values
    result = dict(weight=weights, weight_sh_mean=control_mean, weight_sh_std=control_std,
        weight_sh_95=control_95, has_weights=np.asarray(has_weights),
        has_permutation_weights=np.asarray(has_weights and permutations > 0),
        acc=np.asarray(np.mean(prediction == truth)),
        acc_sh=np.asarray(np.mean(shuffled == truth[:, None]) if permutations else np.nan),
        y_pred=prediction.astype(int), y_test=truth.astype(int), y_pred_sh=shuffled.astype(int),
        n_iterations=np.asarray(iterations), n_permutations=np.asarray(permutations))
    return result


def load_decoding(dataset, method, component, classifier, *, required=False):
    key = (dataset, method.upper(), PARAMETERS['method_pls'], PARAMETERS['data_aug_method'], int(component), classifier)
    if key not in RESULTS:
        _MISSING_RESULTS.add(key)
        if required or STRICT_MISSING_RESULTS:
            raise FileNotFoundError(f'Not computed for {PAIRING_ID}: {key}')
        return None
    record = RESULTS[key]
    signature = {'format_version': CLEANED_FORMAT_VERSION, 'csv': source_signature(record['csv']),
                 'network': source_signature(record['npz'])}
    identity = json.dumps(signature, sort_keys=True)
    if identity in _RESULT_CACHE:
        result = _RESULT_CACHE.pop(identity)
        _RESULT_CACHE[identity] = result
        return record, result
    cleaned = record['csv'].with_name(record['csv'].stem + '_cleaned.npz')
    result = None
    if USE_CLEANED_CACHE and cleaned.is_file() and not RECOMPUTE_CLEANED_DECODING:
        with np.load(cleaned, allow_pickle=False) as saved:
            if json.loads(str(saved['source_signature'].item())) == signature:
                result = {k: saved[k].copy() for k in saved.files if k != 'source_signature'}
    if result is None:
        result = clean_current_csv(record)
        if USE_CLEANED_CACHE:
            temporary = cleaned.with_suffix('.npz.tmp')
            with temporary.open('wb') as stream:
                np.savez_compressed(stream, source_signature=json.dumps(signature, sort_keys=True), **result)
            temporary.replace(cleaned)
    _RESULT_CACHE[identity] = result
    while len(_RESULT_CACHE) > MAX_IN_MEMORY_RESULTS:
        _RESULT_CACHE.popitem(last=False)
    return record, result

In [ ]:
# The single-component overview follows the JSON settings.
band = PARAMETERS['band']
method_pca = PARAMETERS['method_pls']
data_aug_method = PARAMETERS['data_aug_method']
PC_use = PARAMETERS['pc_use']
model_name = PARAMETERS['model']
dim_red_method = PARAMETERS['method'].upper()

print(CONFIG['command'] + ' --pairing ' + PAIRING_ID)
print('The comparison figures retain their original model/component ranges.')
print('Missing combinations require additional runs with those JSON settings; compatible archived runs are included.')

In [ ]:
# Same two-modality overview: classifier weights plus network time courses.
PC_use, model_name, dim_red_method = PARAMETERS['pc_use'], PARAMETERS['model'], PARAMETERS['method'].upper()
fig, ax = plt.subplots(2, 1, figsize=(20, 10))
fig.suptitle(f'{model_name} -- Compo {PC_use+1} -- {dim_red_method}')

for d, dataset_ in enumerate(['ieeg', 'meg']):
    loaded = load_decoding(dataset_, dim_red_method, PC_use, model_name)
    if loaded is None:
        ax[d].text(.5, .5, f'Not computed: {dataset_}, {dim_red_method}, PC {PC_use+1}, {model_name}', ha='center', transform=ax[d].transAxes)
        continue
    record, result = loaded
    train = record['network']['train_condition_mean']
    network_time = record['network']['times']
    time = network_time[record['network']['decoding_time_ids']]
    n_perm, n_iter = int(result['n_permutations']), int(result['n_iterations'])
    if bool(result['has_weights']):
        weight_mean = np.abs(result['weight']).mean(0)
        weight_std = np.abs(result['weight']).std(0)
        ax[d].plot(time, weight_mean, color=modality[dataset_])
        ax[d].fill_between(time, weight_mean-weight_std, weight_mean+weight_std, alpha=.2, color=modality[dataset_])
        if bool(result['has_permutation_weights']):
            weight_sh_mean, weight_sh_std = result['weight_sh_mean'], result['weight_sh_std']
            ax[d].plot(time, weight_sh_mean, color='black', alpha=.5, label=f'Mean {n_perm}-perm model over {n_iter} iter')
            ax[d].fill_between(time, abs(weight_sh_mean), abs(weight_sh_mean)+weight_sh_std, alpha=.1, color='black', label=f'Sd {n_perm}-perm model')
            ax[d].plot(time, result['weight_sh_95'], alpha=1, ls='--', linewidth=.6, color='black', label=f'95% percentil {n_perm}-perm model')
    else:
        ax[d].text(.02, .92, 'This classifier has no linear time weights.', transform=ax[d].transAxes)
    ax[d].set_ylabel('Classifier weights')
    ax[d].grid()
    ax[d].set_title(dataset_ + ' - Accuracy: ' + str(100*np.round(float(result['acc']), 2)) + '%')
    axt = ax[d].twinx()
    axt.set_ylabel('Network activity')
    axt.plot(network_time, train[0, :], ls=':', lw=1, color='blue', label='Network TS')
    axt.plot(network_time, train[1, :], ls=':', lw=1, color='red')

In [ ]:
# Same six-component PCA grid; absent components are labeled, never imputed.
dim_red_method, method_pca, data_aug_method, model_name = 'PCA', PARAMETERS['method_pls'], PARAMETERS['data_aug_method'], 'LR'
fig, ax = plt.subplots(6, 2, figsize=(15, 20))
fig.suptitle(model_name + ' ' + dim_red_method, y=.9)

for PC_use in range(6):
    for d, dataset_ in enumerate(['ieeg', 'meg']):
        loaded = load_decoding(dataset_, dim_red_method, PC_use, model_name)
        if loaded is None:
            ax[PC_use][d].text(.5, .5, f'Not computed: {dataset_}, PCA PC {PC_use+1}', ha='center', transform=ax[PC_use][d].transAxes)
            continue
        record, result = loaded
        axt = ax[PC_use][d].twinx()
        network_time = record['network']['times']
        time = network_time[record['network']['decoding_time_ids']]
        train = record['network']['train_condition_mean']
        ax[PC_use][d].set_ylabel('Network activity')
        ax[PC_use][d].plot(network_time, train[0, :], lw=.5, color='blue', alpha=1, label='Network TS')
        ax[PC_use][d].plot(network_time, train[1, :], lw=.5, color='red', alpha=1)
        ax[PC_use][d].set_title(f'Accuracy: {float(result["acc"])}')
        ax[PC_use][d].grid()
        axt.plot(network_time, train[0, :]-train[1, :], color=modality[dataset_], alpha=.2, lw=.4,
                 label=dataset_+' - Accuracy: '+str(100*np.round(float(result['acc']), 2))+'%')
        if bool(result['has_permutation_weights']):
            weight_mean = np.abs(result['weight']).mean(0)
            axt.plot(time, np.where(weight_mean > result['weight_sh_95'], 1, np.nan), lw=2, color=modality[dataset_])
        axt.set_ylabel('Sign condition difference')

In [ ]:
# Same two projection-method panels, classifier markers, modality colors, and controls.
fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
method_dim_red = {'PCA': ':', 'PLSSVD': '--'}
method_clf = {'LR': 'o', 'RandomForest': 'x', 'SVC_linear': 's'}
for m, a in zip(method_dim_red, ax):
    a.set_title(m)
    for model_name in method_clf:
        for dataset_ in modality:
            acc, acc_s = [], []
            for PC_use in range(8):
                loaded = load_decoding(dataset_, m, PC_use, model_name)
                acc.append(float(loaded[1]['acc']) if loaded is not None else np.nan)
                acc_s.append(float(loaded[1]['acc_sh']) if loaded is not None else np.nan)
            a.scatter(np.arange(len(acc)), acc, marker=method_clf[model_name], color=modality[dataset_], alpha=1, label=model_name+' '+dataset_)
            a.scatter(np.arange(len(acc)), acc_s, marker=method_clf[model_name], color=modality[dataset_], alpha=.5)
            a.set_xticks(np.arange(8), ['PC'+str(i+1) for i in range(8)])
            a.grid()

In [ ]:
# Same accuracy heatmaps: classifier/projection rows and four component columns.
method_dim_red = {'PCA': ':', 'PLSSVD': '--'}
method_clf = {'LR': 'o', 'RandomForest': 'x', 'SVC_linear': 's'}
fig, axes = plt.subplots(1, len(modality), squeeze=False, figsize=(20, 5))
n_rows = len(method_clf)*len(method_dim_red)
for ax, dataset_ in zip(axes.flat, modality):
    labels = []
    values = np.full((n_rows, 4), np.nan)
    for iclf, model_name in enumerate(method_clf):
        for im, m in enumerate(method_dim_red):
            row = iclf*len(method_dim_red)+im
            labels.append(f'{model_name} {m}')
            for PC_use in range(4):
                loaded = load_decoding(dataset_, m, PC_use, model_name)
                if loaded is not None:
                    values[row, PC_use] = float(loaded[1]['acc'])
    sns.heatmap(values, mask=~np.isfinite(values), yticklabels=labels, xticklabels=range(4), ax=ax,
                vmin=.5, vmax=1, cmap='Reds', annot=True)
    if not np.isfinite(values).any():
        ax.text(.5, .5, 'No matching results computed', ha='center', transform=ax.transAxes)
    ax.set_title(dataset_)
    ax.set_xlabel('PC_use')
plt.tight_layout()

In [ ]:
# Same overlaid LR weight-threshold traces for five components.
model_name = 'LR'
method_dim_red = {'PCA': ':', 'PLSSVD': '--'}
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
fig.suptitle(model_name)
for im, m in enumerate(method_dim_red):
    ax[im].set_title(m)
    for d, dataset_ in enumerate(modality):
        for PC_use in range(5):
            loaded = load_decoding(dataset_, m, PC_use, model_name)
            if loaded is None or not bool(loaded[1]['has_permutation_weights']):
                continue
            record, result = loaded
            time = record['network']['times'][record['network']['decoding_time_ids']]
            weight_mean = np.abs(result['weight']).mean(0)
            weight_thr = np.where(weight_mean > result['weight_sh_95'], PC_use+d*.1, np.nan)
            ax[im].plot(time, weight_thr, color=modality[dataset_])
for a in ax:
    a.grid()

if _MISSING_RESULTS:
    print('The following method/component/classifier combinations have not been computed for the selected pairing:')
    display(pd.DataFrame(sorted(_MISSING_RESULTS), columns=['modality', 'method', 'representation', 'augmentation', 'pc_use', 'classifier']))